# Split Synthetic Images into Train and Test Sets

This notebook reads all CSV files in the dataset `Metadata` directory, uses `synthetic_split.csv` to determine each image's split, and copies the images into a new `Synthetic_split/train` and `Synthetic_split/test` directory. The original `Synthetic` directory is unchanged.

In [ ]:
from pathlib import Path
import shutil

import pandas as pd

# The notebook is expected to run from the repository root.
REPO_ROOT = Path.cwd()
DATASET_ROOT = (
    REPO_ROOT
    / "data"
    / "f5ddba02-4295-48d9-8ad5-ba8583871323"
    / "bc6ba36c-aa02-4630-a0ea-4150b9b69e36"
)
METADATA_DIR = DATASET_ROOT / "Metadata"
SYNTHETIC_DIR = DATASET_ROOT / "Synthetic"
OUTPUT_DIR = DATASET_ROOT / "Synthetic_split"

if not METADATA_DIR.is_dir():
    raise FileNotFoundError(f"Metadata directory not found: {METADATA_DIR}")
if not SYNTHETIC_DIR.is_dir():
    raise FileNotFoundError(f"Synthetic image directory not found: {SYNTHETIC_DIR}")

print(f"Dataset root: {DATASET_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")

## Read Metadata CSV Files

All CSV files in `Metadata` are loaded into a dictionary keyed by filename stem. The split file is selected by its filename rather than relying on a particular CSV ordering.

In [ ]:
metadata_files = sorted(METADATA_DIR.glob("*.csv"))
if not metadata_files:
    raise FileNotFoundError(f"No CSV files found in {METADATA_DIR}")

metadata_tables = {path.stem: pd.read_csv(path) for path in metadata_files}
print("Loaded metadata files:")
for name, table in metadata_tables.items():
    print(f"  {name}: {table.shape[0]:,} rows x {table.shape[1]} columns")

if "synthetic_split" not in metadata_tables:
    raise KeyError("Expected Metadata/synthetic_split.csv")

split_df = metadata_tables["synthetic_split"].copy()
required_columns = {"s3_path", "split"}
missing_columns = required_columns.difference(split_df.columns)
if missing_columns:
    raise KeyError(f"synthetic_split.csv is missing columns: {sorted(missing_columns)}")

split_df[["s3_path", "split"]].head()

In [ ]:
# Normalize split labels and use only the image basename from each metadata path.
split_df["split"] = split_df["split"].astype(str).str.strip().str.lower()
split_df["image_name"] = split_df["s3_path"].map(lambda value: Path(str(value)).name)

allowed_splits = {"train", "test"}
unexpected_splits = sorted(set(split_df["split"]) - allowed_splits)
if unexpected_splits:
    raise ValueError(f"Unexpected split labels: {unexpected_splits}. Expected train/test.")

duplicate_names = split_df["image_name"][split_df["image_name"].duplicated()].unique().tolist()
if duplicate_names:
    raise ValueError(f"Images assigned more than once in synthetic_split.csv: {duplicate_names[:5]}")

missing_images = [
    image_name
    for image_name in split_df["image_name"]
    if not (SYNTHETIC_DIR / image_name).is_file()
]
if missing_images:
    raise FileNotFoundError(
        f"{len(missing_images)} images from synthetic_split.csv were not found in Synthetic."
        f" Examples: {missing_images[:5]}"
    )

print(split_df["split"].value_counts().sort_index())
print(f"Validated {len(split_df):,} split assignments and confirmed all source images exist.")

## Copy Images into Train and Test Directories

The output directory is created separately from the source directory. Existing files with the same names are overwritten so the notebook can be rerun safely.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

copied_counts = {split_name: 0 for split_name in sorted(allowed_splits)}
for row in split_df.itertuples(index=False):
    source_path = SYNTHETIC_DIR / row.image_name
    destination_dir = OUTPUT_DIR / row.split
    destination_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_path, destination_dir / row.image_name)
    copied_counts[row.split] += 1

print("Copied images:")
for split_name, count in copied_counts.items():
    print(f"  {split_name}: {count:,}")

In [ ]:
# Verify the output counts and report the resulting directories.
output_counts = {
    split_name: len(list((OUTPUT_DIR / split_name).glob("*")))
    for split_name in sorted(allowed_splits)
}

if output_counts != copied_counts:
    raise RuntimeError(f"Output verification failed: expected {copied_counts}, found {output_counts}")

print("Split complete and verified.")
print(f"Train images: {OUTPUT_DIR / 'train'}")
print(f"Test images:  {OUTPUT_DIR / 'test'}")